In [ ]:
import json

# 1. 샘플 데이터 로드
mock_data = {
  "name": "project-alpha",
  "description": "None", 
  "language": "Python",
  "topics": [],
  "selectedFileContents": [
    {
      "path": "requirements.txt",
      "content": "pandas==2.0.0\nyfinance==0.2.18\nschedule==1.2.0\nrequests==2.28.2"
    },
    {
      "path": "main.py",
      "content": "import schedule\nimport time\nfrom src.collector import fetch_data\nfrom src.notifier import send_msg\n\ndef job():\n    price = fetch_data('AAPL')\n    if price > 150:\n        send_msg(f'Apple target hit: {price}')\n\nschedule.every().hour.do(job)\nwhile True:\n    schedule.run_pending()\n    time.sleep(1)"
    },
    {
      "path": "src/collector.py",
      "content": "import yfinance as yf\n\ndef fetch_data(ticker):\n    data = yf.Ticker(ticker)\n    return data.history(period='1d')['Close'].iloc[-1]"
    },
    {
      "path": "src/notifier.py",
      "content": "import requests\n\ndef send_msg(text):\n    url = f'https://api.telegram.org/bot{TOKEN}/sendMessage'\n    requests.post(url, data={'chat_id': ID, 'text': text})"
    }
  ]
}

In [ ]:
import requests
import json
import time
import os
from dotenv import load_dotenv

# 1. 설정
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# 비교하고 싶은 모델 리스트
MODELS = {
    "Qwen2.5-Coder-7B": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Llama-3.2-8B": "meta-llama/Llama-3.2-8B-Instruct",
    "DeepSeek-Coder-V2": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
}

def call_hf_api(model_id, prompt):
    api_url = f"https://api-inference.huggingface.co/models/{model_id}"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    
    payload = {
        "inputs": f"<|system|>\nYou are a helpful assistant expert in code analysis.\n<|user|>\n{prompt}\n<|assistant|>\n",
        "parameters": {"max_new_tokens": 1000, "return_full_text": False}
    }
    
    response = requests.post(api_url, headers=headers, json=payload)
    
    if response.status_code == 200:
        return response.json()[0]['generated_text']
    else:
        return f"Error: {response.status_code}, {response.text}"

def test_model_inference(model_alias, model_id, data):
    print(f"\n" + "="*50)
    print(f"🚀 모델 테스트 중: {model_alias}")
    print("="*50)
    
    context = json.dumps(data, indent=2, ensure_ascii=False)
    prompt = f"""
다음은 특정 GitHub 저장소에서 추출한 정보와 코드의 일부다.
이 정보를 분석해서 다음 두 질문에 답하시오. 답변은 반드시 한국어로 작성하시오.

1. 이 프로젝트는 한 문장으로 정의하면 무엇을 하는 프로젝트인가?
2. 이 프로젝트의 핵심 기능 3가지는 무엇인가?
이 두가지를 코드 기반으로 추론해서 리스트로 작성하시오.

[데이터]
{context}
"""
    
    start_time = time.time()
    result = call_hf_api(model_id, prompt)
    end_time = time.time()
    
    print(result)
    print(f"\n⏱️ 소요 시간: {end_time - start_time:.2f}초")
    return result

# 3. 실행 및 비교
results = {}
for alias, model_id in MODELS.items():
    results[alias] = test_model_inference(alias, model_id, mock_data)